# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Corpus:** Gloomhaven rulebook (`gloomhaven_rulebook.pdf`)

Two pipelines over the *identical* chunked corpus. Only the selection step differs:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-Ranked) |
|---|---|---|
| Retrieve | Bi-encoder, cosine, Top 10 | *the same Top 10* |
| Select | Top 3 as-is | Cross-encoder re-sort → Top 3 |
| Generate | Answer 1 | Answer 2 (identical prompt) |

Same chunks, same prompt, same LLM — so any difference in the answers is attributable
to ranking alone.

## 1. Setup & Environment

In [1]:
import os, re, time, textwrap
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF_PATH = "gloomhaven_rulebook.pdf"
BI_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CHUNK_SIZE, OVERLAP = 550, 100
TOP_K, TOP_N = 10, 3   # bi-encoder candidates, chunks shown to the LLM

# Key read from the environment so the notebook is safe to push to a public repo.
#   export GEMINI_API_KEY="..."      (set it BEFORE launching Jupyter)
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
print("Gemini key found" if GEMINI_API_KEY else "No key — retrieval cells still run; LLM cells will not")

2026-10-16 01:26:50.287213: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-16 01:26:50.326711: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-10-16 01:26:51.419165: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


/home/sampath/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Gemini key found


## 2. Load & chunk the PDF

Two things this PDF forced:

1. **No blank lines.** `extract_text()` returns 1399 single newlines and *zero* double
   newlines, so the usual "split on `\n\n`" produces one giant chunk. We split on the 34
   ALL-CAPS section headings instead, which also keeps each rule with its own section.
2. **The reference glossary is printed twice.** Duplicate chunks are deduplicated — otherwise
   two identical chunks occupy two of the ten candidate slots.

Chunk size is the knob that silently decides the whole experiment. Too large and every chunk
contains the answer (nothing for re-ranking to fix); too small and an "except when..." clause
is severed from the rule it modifies. 550 was chosen by sweeping 450/550/700 — see section 7.

In [2]:
def load_pdf(path):
    text = "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)
    return re.sub(r"[ \t]+", " ", text)

HEADING = re.compile(r"^(?:\d+\. )?[A-Z][A-Z0-9 &/,'-]{3,}$")
# Front/back matter of the summary PDF — publisher boilerplate, not game rules.
DROP = {"PREAMBLE", "VIDEOS", "PHOTOS", "TWEETS", "NEWS", "MORE GUIDES"}

def sections(text):
    """Split on ALL-CAPS headings -> [(heading, body)]."""
    cur, buf, out = "PREAMBLE", [], []
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            continue
        if HEADING.match(s):
            if buf:
                out.append((cur, " ".join(buf)))
            cur, buf = s, []
        else:
            buf.append(s)
    if buf:
        out.append((cur, " ".join(buf)))
    return out

def chunk(text, size=CHUNK_SIZE, overlap=OVERLAP):
    """Pack each section's sentences up to `size`; prepend heading for provenance."""
    chunks = []
    for head, body in sections(text):
        if head in DROP:
            continue
        # sentence-boundary split so an "except when..." clause is never cut mid-sentence
        buf = ""
        for s in re.split(r"(?<=[.!?]) +", body):
            if len(buf) + len(s) > size and buf:
                chunks.append(f"[{head}] {buf.strip()}")
                buf = buf[-overlap:] + " "
            buf += s + " "
        if buf.strip():
            chunks.append(f"[{head}] {buf.strip()}")
    return list(dict.fromkeys(chunks))   # dedupe, preserving order

RAW = load_pdf(PDF_PATH)
CHUNKS = chunk(RAW)
sz = sorted(map(len, CHUNKS))
print(f"{len(RAW):,} chars -> {len(sections(RAW))} sections -> {len(CHUNKS)} chunks")
print(f"chunk chars  min {sz[0]} / median {sz[len(sz)//2]} / max {sz[-1]}")
print("\n--- sample chunk ---\n", textwrap.fill(CHUNKS[40], 95))

74,916 chars -> 39 sections -> 167 chunks
chunk chars  min 160 / median 490 / max 571

--- sample chunk ---
 [ACTIONS] tion are applied to the target as well, except for effects that would result in
additional targets.  Conditions When a condition is applied to a figure (except Curse and
Bless), place its token on the stat sleeve in the section corresponding to the monster’s
standee number, where it remains until the requirements for removing the effect are met. Only
one of each condition type may be applied to a single figure at a time, though they can be
reapplied to refresh their duration.


## 3. Pipeline 1 — Baseline retrieval (bi-encoder)

In [3]:
bi = SentenceTransformer(BI_MODEL)
# Normalize once -> cosine similarity is a plain dot product.
# 132 chunks does not need FAISS/Chroma; a NumPy matmul is the whole vector store.
EMB = bi.encode(CHUNKS, normalize_embeddings=True, show_progress_bar=True, batch_size=64)
print(EMB.shape)

def retrieve(query, k=TOP_K):
    """Top-k chunk indices by cosine similarity -> [(idx, score), ...]."""
    q = bi.encode([query], normalize_embeddings=True)[0]
    s = EMB @ q
    return [(int(i), float(s[i])) for i in np.argsort(-s)[:k]]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(167, 384)


## 4. Pipeline 2 — Cross-encoder re-ranking

Consumes **the same Top 10** Pipeline 1 produced, and re-scores each `(query, chunk)` pair
jointly rather than comparing two independently-computed vectors.

In [4]:
ce = CrossEncoder(CE_MODEL)

def rerank(query, candidates):
    """candidates: [(idx, bi_score)] -> same items re-sorted by cross-encoder score."""
    scores = ce.predict([(query, CHUNKS[i]) for i, _ in candidates])
    return sorted([(i, float(s)) for (i, _), s in zip(candidates, scores)], key=lambda t: -t[1])

## 5. Generation — one prompt, both pipelines

Deliberately a single function. If the two pipelines used different prompt strings the
comparison would be confounded and the experiment would prove nothing.

In [5]:
PROMPT = """You are answering rules questions about the Gloomhaven board game.
Answer using ONLY the context below. If the context does not contain the answer, say so
explicitly rather than guessing. Quote the specific rule you rely on.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

MODEL = "gemini-3.8-flash"

_client = None
def _get_client():
    global _client
    if _client is None:
        from google import genai
        assert GEMINI_API_KEY, "set GEMINI_API_KEY and restart the kernel"
        _client = genai.Client(api_key=GEMINI_API_KEY)
    return _client

def answer(query, selected, retries=5):
    """selected: [(idx, score)] -> LLM answer string."""
    ctx = "\n\n---\n\n".join(f"[chunk {i}]\n{CHUNKS[i]}" for i, _ in selected)
    prompt = PROMPT.format(context=ctx, question=query)
    for a in range(retries):
        try:
            return _get_client().models.generate_content(
                model=MODEL, contents=prompt).text.strip()
        except Exception as e:
            # the free tier returns transient 503s under load; back off and retry
            if a == retries - 1:
                raise
            time.sleep(2 ** a)


## 6. Running both pipelines + latency instrumentation

In [6]:
def compare(query, gold=None, verbose=True):
    """Run both pipelines on one query. `gold` = substring that marks the correct chunk."""
    t0 = time.perf_counter()
    cands = retrieve(query, TOP_K)            # shared stage-1 for both pipelines
    t_ret = time.perf_counter() - t0

    t0 = time.perf_counter()
    reranked = rerank(query, cands)
    t_ce = time.perf_counter() - t0

    def gold_rank(lst):
        if not gold:
            return None
        for r, (i, _) in enumerate(lst, 1):
            if gold.lower() in CHUNKS[i].lower():
                return r
        return None

    res = dict(query=query, gold=gold, cands=cands, base=cands[:TOP_N],
               reranked=reranked, top_re=reranked[:TOP_N],
               bi_rank=gold_rank(cands), ce_rank=gold_rank(reranked),
               t_retrieval=t_ret, t_crossencoder=t_ce)

    if verbose:
        bi_pos = {i: r for r, (i, _) in enumerate(cands, 1)}
        print(f"Q: {query}")
        print(f"\n  gold chunk: bi-rank #{res['bi_rank']}  ->  ce-rank #{res['ce_rank']}")
        print(f"\n  {'chunk':>6} {'bi':>4} {'ce':>4}  {'bi_score':>9} {'ce_score':>9}")
        for r, (i, s) in enumerate(reranked[:TOP_N], 1):
            mark = "  <- promoted into top 3" if bi_pos[i] > TOP_N else ""
            bs = dict(cands)[i]
            print(f"  {i:>6} {bi_pos[i]:>4} {r:>4}  {bs:>9.3f} {s:>9.3f}{mark}")
        print(f"\n  retrieval {t_ret*1000:.1f} ms | cross-encoder {t_ce*1000:.1f} ms")
    return res

def show_chunks(res):
    """Eyeball the actual text each pipeline would hand the LLM."""
    for label, sel in (("BASELINE top 3", res["base"]), ("RE-RANKED top 3", res["top_re"])):
        print(f"\n##### {label}")
        for i, s in sel:
            hit = "  *** contains the answer" if res["gold"] and res["gold"].lower() in CHUNKS[i].lower() else ""
            print(f"\n[chunk {i}] score={s:.3f}{hit}\n{textwrap.fill(CHUNKS[i][:400], 95)}")

def show_answers(res):
    """Generate Answer 1 vs Answer 2 for an already-compared query."""
    res["answer_1"] = answer(res["query"], res["base"])
    res["answer_2"] = answer(res["query"], res["top_re"])
    for label in ("answer_1", "answer_2"):
        tag = "ANSWER 1 (baseline top 3)" if label == "answer_1" else "ANSWER 2 (re-ranked top 3)"
        print(f"\n===== {tag} =====")
        print(textwrap.fill(res[label], 95))
    return res

## 7. Failure Mode Analysis (stress testing)

Queries were built by reading the rulebook for genuinely tricky clauses first, then writing
the question that *should* retrieve each one — deliberately phrased in **synonyms** rather
than the rulebook's own vocabulary ("strike back" for *retaliate*, "creature I conjured" for
*summon*, "vitality" for *HP*). That synonym gap is exactly what a bi-encoder struggles with.

`gold` records a substring that must appear in a chunk for that chunk to actually answer the
question, which makes "did retrieval work" measurable instead of a judgement call.

The failing and non-failing cases are both kept below — including two where re-ranking made
things *worse* and two the baseline never surfaced at all.

In [7]:
QUERIES = [
    # --- re-ranking rescues these (the headline cases) ---
    ("keyword_trap", "am I allowed to skip part of what my card tells me to do?",
     "free to not perform any part of the action"),
    ("negation/exception", "may a stunned figure drink a healing potion on its turn?",
     "cannot perform any abilities or use items except"),
    ("buried", "what occurs if a monster has nothing it is able to aim at?",
     "no valid targets to focus on"),
    ("buried", "may I stockpile two of the same piece of equipment?",
     "cannot own more than 1 copy of any item"),

    # --- controls: baseline already correct, re-ranking keeps it there ---
    ("control", "does mending someone's injuries also cure the venom afflicting them?",
     "removes the poison token"),
    ("control", "can a beast walk through a shut entryway the same way a hero can?",
     "act as a wall for any monsters"),

    # --- honest negatives: re-ranking demotes a correct chunk out of the top 3 ---
    ("regression", "do I get a bigger hand of cards as I grow more powerful?",
     "never changes your maximum hand size"),
    ("regression", "do sealed entrances open when someone steps onto them?",
     "act as walls until the conditions to open them"),

    # --- honest negatives: stage 1 never surfaced it, so re-ranking cannot help ---
    ("stage1_miss", "if I strike back at someone who hit me, does that count as attacking them?",
     "retaliate is not an attack"),
    ("stage1_miss", "is there a ceiling on restoring a wounded hero's vitality?",
     "cannot have more HPs than your maximum"),
]

RESULTS = {}
for tag, q, gold in QUERIES:
    key = f"{tag}:{q[:35]}"
    print(f"\n{'='*95}\n[{tag}]")
    RESULTS[key] = compare(q, gold)


[keyword_trap]
Q: am I allowed to skip part of what my card tells me to do?

  gold chunk: bi-rank #7  ->  ce-rank #1

   chunk   bi   ce   bi_score  ce_score
      17    7    1      0.330    -3.475  <- promoted into top 3
      11    2    2      0.385    -6.301
      85    5    3      0.359    -6.347  <- promoted into top 3

  retrieval 7.7 ms | cross-encoder 160.0 ms

[negation/exception]


Q: may a stunned figure drink a healing potion on its turn?

  gold chunk: bi-rank #6  ->  ce-rank #2

   chunk   bi   ce   bi_score  ce_score
      42    2    1      0.367    -0.715
      43    6    2      0.344    -1.680  <- promoted into top 3
     143    4    3      0.351    -2.001  <- promoted into top 3

  retrieval 6.8 ms | cross-encoder 178.5 ms

[buried]
Q: what occurs if a monster has nothing it is able to aim at?

  gold chunk: bi-rank #7  ->  ce-rank #3

   chunk   bi   ce   bi_score  ce_score
      72    6    1      0.478     0.820  <- promoted into top 3
      73    3    2      0.510    -0.300
      71    7    3      0.476    -1.628  <- promoted into top 3

  retrieval 7.7 ms | cross-encoder 148.3 ms

[buried]


Q: may I stockpile two of the same piece of equipment?

  gold chunk: bi-rank #4  ->  ce-rank #3

   chunk   bi   ce   bi_score  ce_score
     121    2    1      0.203   -11.002
      50    3    2      0.188   -11.027
     153    4    3      0.168   -11.313  <- promoted into top 3

  retrieval 9.4 ms | cross-encoder 138.1 ms

[control]
Q: does mending someone's injuries also cure the venom afflicting them?

  gold chunk: bi-rank #1  ->  ce-rank #1

   chunk   bi   ce   bi_score  ce_score
     143    1    1      0.385    -6.423
      42    2    2      0.312    -8.171
      41    4    3      0.271    -9.544  <- promoted into top 3

  retrieval 8.6 ms | cross-encoder 137.7 ms

[control]


Q: can a beast walk through a shut entryway the same way a hero can?

  gold chunk: bi-rank #2  ->  ce-rank #1

   chunk   bi   ce   bi_score  ce_score
      88    2    1      0.411    -4.690
      69    6    2      0.333    -7.699  <- promoted into top 3
      75    3    3      0.403    -7.711

  retrieval 7.3 ms | cross-encoder 162.4 ms

[regression]
Q: do I get a bigger hand of cards as I grow more powerful?

  gold chunk: bi-rank #2  ->  ce-rank #6

   chunk   bi   ce   bi_score  ce_score
     127    1    1      0.529    -3.682
     126    7    2      0.333    -5.431  <- promoted into top 3
     128    3    3      0.415    -5.448

  retrieval 8.6 ms | cross-encoder 135.6 ms

[regression]


Q: do sealed entrances open when someone steps onto them?

  gold chunk: bi-rank #2  ->  ce-rank #4

   chunk   bi   ce   bi_score  ce_score
     163    4    1      0.410    -3.763  <- promoted into top 3
      88    1    2      0.553    -4.168
     162    3    3      0.441    -5.313

  retrieval 7.1 ms | cross-encoder 148.3 ms

[stage1_miss]


Q: if I strike back at someone who hit me, does that count as attacking them?

  gold chunk: bi-rank #None  ->  ce-rank #None

   chunk   bi   ce   bi_score  ce_score
      76    7    1      0.351    -5.045  <- promoted into top 3
      54    3    2      0.377    -5.933
      35    1    3      0.406    -6.032

  retrieval 8.5 ms | cross-encoder 208.0 ms

[stage1_miss]


Q: is there a ceiling on restoring a wounded hero's vitality?

  gold chunk: bi-rank #None  ->  ce-rank #None

   chunk   bi   ce   bi_score  ce_score
      42    3    1      0.281    -9.215
     143    6    2      0.258   -10.258  <- promoted into top 3
     157    1    3      0.306   -10.832

  retrieval 10.2 ms | cross-encoder 199.0 ms


### 7a. Rank-shift summary

In [8]:
def verdict(r):
    b, c = r["bi_rank"], r["ce_rank"]
    if b is None:            return "stage-1 miss"
    if b > TOP_N and c and c <= TOP_N:  return "FIXED by re-rank"
    if b <= TOP_N and (c is None or c > TOP_N): return "BROKEN by re-rank"
    if b <= TOP_N:           return "both correct"
    return "still buried"

print(f"{'verdict':<18} {'bi':>4} {'ce':>4}  query")
print("-" * 95)
for k, r in RESULTS.items():
    print(f"{verdict(r):<18} {str(r['bi_rank']):>4} {str(r['ce_rank']):>4}  {r['query'][:62]}")

n_fix = sum(verdict(r) == "FIXED by re-rank" for r in RESULTS.values())
n_brk = sum(verdict(r) == "BROKEN by re-rank" for r in RESULTS.values())
print(f"\nre-ranking fixed {n_fix}, broke {n_brk}, out of {len(RESULTS)} queries")

verdict              bi   ce  query
-----------------------------------------------------------------------------------------------
FIXED by re-rank      7    1  am I allowed to skip part of what my card tells me to do?
FIXED by re-rank      6    2  may a stunned figure drink a healing potion on its turn?
FIXED by re-rank      7    3  what occurs if a monster has nothing it is able to aim at?
FIXED by re-rank      4    3  may I stockpile two of the same piece of equipment?
both correct          1    1  does mending someone's injuries also cure the venom afflicting
both correct          2    1  can a beast walk through a shut entryway the same way a hero c
BROKEN by re-rank     2    6  do I get a bigger hand of cards as I grow more powerful?
BROKEN by re-rank     2    4  do sealed entrances open when someone steps onto them?
stage-1 miss       None None  if I strike back at someone who hit me, does that count as att
stage-1 miss       None None  is there a ceiling on restoring a wounded

### 7b. Chunk size sweep

Evidence that chunk size, not model quality, drives a lot of the apparent result — worth a
sentence in the report.

In [9]:
_orig_chunks, _orig_emb = CHUNKS, EMB
print(f"{'size':>5} {'chunks':>7} {'fixed':>6} {'broken':>7} {'stage-1 misses':>15}")
for size in (450, 550, 700):
    CHUNKS = chunk(RAW, size=size, overlap=OVERLAP)
    EMB = bi.encode(CHUNKS, normalize_embeddings=True, batch_size=64)
    vs = [verdict(compare(q, g, verbose=False)) for _, q, g in QUERIES]
    print(f"{size:>5} {len(CHUNKS):>7} {vs.count('FIXED by re-rank'):>6} "
          f"{vs.count('BROKEN by re-rank'):>7} {vs.count('stage-1 miss'):>15}")
CHUNKS, EMB = _orig_chunks, _orig_emb   # restore the 550 corpus

 size  chunks  fixed  broken  stage-1 misses


  450     219      2       0               3


  550     167      4       2               2


  700     126      1       1               2


### 7c. Inspect the chunks for one rescued query

This is where you confirm the cross-encoder surfaced the genuinely correct rule rather than
getting lucky.

In [10]:
# first query that re-ranking rescued
_k = next(k for k, r in RESULTS.items() if verdict(r) == "FIXED by re-rank")
print(_k)
print(RESULTS[_k]["query"])
show_chunks(RESULTS[_k])

keyword_trap:am I allowed to skip part of what m
am I allowed to skip part of what my card tells me to do?

##### BASELINE top 3

[chunk 18] score=0.396
[3. CHARACTER AND MONSTER TURNS] ack 2 action on the top half or a Move 2 action on the bottom.
Always discard a card used this way.  On your turn, before, during, or after performing your 2
actions, you can use any number of items you have equipped.

[chunk 11] score=0.385
[1. CARD SELECTION] Each player secretly selects 2 cards from their hand to play facedown in
front of them. One should be selected as the leading card which determines the player’s
initiative. Players should not share specifics about their cards, but may discuss strategy and
make general statements about their actions for the round. Alternatively, you may perform a
long rest. If you have one, or n

[chunk 16] score=0.373
[3. CHARACTER AND MONSTER TURNS] On your turn, perform the top action of one of the 2 ability
cards played and the bottom action of the other. It d

### 7d. Answer 1 vs Answer 2

Requires `GEMINI_API_KEY`. Only the queries where ranking actually changed the top 3 are
worth spending LLM calls on.

In [11]:
for k, r in RESULTS.items():
    if verdict(r) in ("FIXED by re-rank", "BROKEN by re-rank"):
        print(f"\n{'='*95}\n[{k}]\n{r['query']}")
        show_answers(r)


[keyword_trap:am I allowed to skip part of what m]
am I allowed to skip part of what my card tells me to do?



===== ANSWER 1 (baseline top 3) =====
The provided context does not contain the answer to whether you are allowed to skip part of
what your card tells you to do.   The closest rule provided states only that: > "The abilities
of the action must be done in the order written and can’t be interrupted by the other card’s
action." (chunk 16)  However, it does not mention whether any parts or abilities of an action
can be skipped.

===== ANSWER 2 (re-ranked top 3) =====
Yes, you are allowed to skip parts of an action, unless that part causes a negative effect on
yourself or your allies.  As stated in the rules: > "You are free to not perform any part of
the action, but you must perform any part that causes a negative effect on yourself or your
allies." (chunk 17)

[negation/exception:may a stunned figure drink a healin]
may a stunned figure drink a healing potion on its turn?



===== ANSWER 1 (baseline top 3) =====
Based on the provided context, the answer is not contained within the text. The context does
not mention the "stunned" condition or rules regarding drinking potions.

===== ANSWER 2 (re-ranked top 3) =====
No, a stunned figure cannot drink a healing potion on its turn.   According to chunk 43: >
"STUN The figure cannot perform any abilities or use items except, if a character, to perform a
long rest."

[buried:what occurs if a monster has nothin]
what occurs if a monster has nothing it is able to aim at?



===== ANSWER 1 (baseline top 3) =====
Based on the provided context, the answer is not mentioned. The context explains how a monster
determines a focus and how it behaves when it cannot move into attack range of its focus, but
it does not state what occurs if a monster has nothing it is able to aim at or focus on.

===== ANSWER 2 (re-ranked top 3) =====
Based on the provided context, if a monster has no valid targets to focus on, it does not move
or attack, but it still performs any other actions on its ability card that it can.  Specific
rule quote: > "Where there are no valid targets to focus on, regardless of the number of hexes
it could move, a monster does not move or attack, but performs any other actions on its card
that it can." [chunk 71]

[buried:may I stockpile two of the same pie]
may I stockpile two of the same piece of equipment?



===== ANSWER 1 (baseline top 3) =====
No, you may not.   According to chunk 121, **"No character may own duplicates of the same
item."**

===== ANSWER 2 (re-ranked top 3) =====
No, you may not.   According to the rules: * "No character may own duplicates of the same
item." [chunk 121] * "You cannot own more than 1 copy of any item card." [chunk 153]

[regression:do I get a bigger hand of cards as ]
do I get a bigger hand of cards as I grow more powerful?


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.8-flash\nPlease retry in 58.044076583s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flash'

## 8. Latency profiling

In [12]:
print(f"{'query':<42} {'baseline':>10} {'+rerank':>10} {'overhead':>10}")
print("-" * 95)
tb = tr = 0.0
for k, r in RESULTS.items():
    b = r["t_retrieval"] * 1000
    t = b + r["t_crossencoder"] * 1000
    tb += b; tr += t
    print(f"{r['query'][:40]:<42} {b:>8.1f}ms {t:>8.1f}ms {t/b:>9.1f}x")
n = len(RESULTS)
print("-" * 95)
print(f"{'MEAN':<42} {tb/n:>8.1f}ms {tr/n:>8.1f}ms {tr/tb:>9.1f}x")

query                                        baseline    +rerank   overhead
-----------------------------------------------------------------------------------------------
am I allowed to skip part of what my car        7.7ms    167.7ms      21.8x
may a stunned figure drink a healing pot        6.8ms    185.4ms      27.1x
what occurs if a monster has nothing it         7.7ms    156.0ms      20.3x
may I stockpile two of the same piece of        9.4ms    147.5ms      15.6x
does mending someone's injuries also cur        8.6ms    146.3ms      17.0x
can a beast walk through a shut entryway        7.3ms    169.7ms      23.4x
do I get a bigger hand of cards as I gro        8.6ms    144.2ms      16.8x
do sealed entrances open when someone st        7.1ms    155.3ms      21.9x
if I strike back at someone who hit me,         8.5ms    216.5ms      25.4x
is there a ceiling on restoring a wounde       10.2ms    209.2ms      20.6x
--------------------------------------------------------------------

### 8a. Scaling evidence — why not cross-encode everything?

In [13]:
probe = QUERIES[0][1]

t0 = time.perf_counter(); bi.encode([probe], normalize_embeddings=True); t_embed = time.perf_counter() - t0
t0 = time.perf_counter(); ce.predict([(probe, c) for c in CHUNKS[:50]]); per_pair = (time.perf_counter() - t0) / 50

print(f"bi-encoder : {t_embed*1000:.1f} ms to embed the query, then one matmul")
print(f"             (chunk vectors are precomputed ONCE and reused for every query)")
print(f"cross-encoder: {per_pair*1000:.1f} ms per (query, chunk) pair, computable only at query time\n")
for n, label in ((TOP_K, f"top-{TOP_K} rerank (what we do)"),
                 (len(CHUNKS), f"this corpus ({len(CHUNKS)} chunks)"),
                 (100_000, "a 100k-chunk corpus"),
                 (1_000_000, "a 1M-chunk corpus")):
    s = per_pair * n
    print(f"  {label:<34} {s:>10.2f} s" + (f"  ({s/3600:.1f} hours)" if s > 3600 else ""))

bi-encoder : 7.8 ms to embed the query, then one matmul
             (chunk vectors are precomputed ONCE and reused for every query)
cross-encoder: 14.6 ms per (query, chunk) pair, computable only at query time

  top-10 rerank (what we do)               0.15 s
  this corpus (167 chunks)                 2.44 s
  a 100k-chunk corpus                   1458.66 s
  a 1M-chunk corpus                    14586.64 s  (4.1 hours)


## 9. Report notes

> The submitted report must be in your own words — the brief explicitly says it should not be
> AI-generated. These are pointers to the evidence, not prose to copy.

**1. Architecture comparison.** Bi-encoder embeds query and chunk *independently*; the chunk
vector is frozen before it has ever seen the query, so matching is a dot product between two
summaries. A cross-encoder runs joint self-attention over `[query || chunk]` together, so every
query token can attend to every chunk token. See section 3 vs 4.

**2. Rank-shift analysis.** Tables in 7a; per-query rank moves and both scores printed in 7.
Compare Answer 1 vs Answer 2 from 7d — look specifically at whether the baseline answered
confidently from a chunk that did not contain the governing exception.

**3. Latency profiling.** Section 8.

**4. Scaling question.** Section 8a is the measured evidence. The bi-encoder's chunk vectors
are computed once and reused; the cross-encoder cannot precompute anything, because its score
depends on the query and the chunk *jointly*. Cost therefore grows linearly with corpus size
at query time. Retrieve-then-rerank pays cross-encoder cost on 10 pairs instead of N.

**Worth mentioning (the honest findings):** re-ranking is not a free win. Section 7a shows
cases where it demoted a correct chunk, and two where stage-1 retrieval never surfaced the
right chunk at all — a re-ranker can only reorder what it is given, so recall@10 is a hard
ceiling on the whole pipeline. Section 7b shows chunk size moving the results as much as the
models do.